In [ ]:
import os
import sys
from mpi4py import MPI
import pickle
# torch
import torch

# quimb
import quimb.tensor as qtn
import autoray as ar

from vmc_torch.experiment.tn_model import fTNModel_reuse, fTN_BFA_cluster_Model_reuse
from vmc_torch.experiment.tn_model import init_weights_to_zero
from vmc_torch.sampler import MetropolisExchangeSamplerSpinful_2D_reusable
from vmc_torch.variational_state import Variational_State
from vmc_torch.optimizer import SGD, SR, DecayScheduler
from vmc_torch.VMC import VMC
from vmc_torch.hamiltonian_torch import spinful_Fermi_Hubbard_square_lattice_torch
from vmc_torch.torch_utils import SVD,QR_tao
from vmc_torch.utils import closest_divisible
from symmray.fermionic_local_operators import FermionicOperator
# Register safe SVD and QR functions to torch
ar.register_function('torch','linalg.svd',SVD.apply)
ar.register_function('torch','linalg.qr',QR_tao.apply)
pwd = '/home/sijingdu/TNVMC/VMC_code/vmc_torch/data'



COMM = MPI.COMM_WORLD
SIZE = COMM.Get_size()
RANK = COMM.Get_rank()
# Hamiltonian parameters
Lx = int(8)
Ly = int(8)
symmetry = 'U1SU'
t = 1.0
U = 8.0
N_f = int(Lx*Ly-8)
n_fermions_per_spin = (N_f//2, N_f//2)
H = spinful_Fermi_Hubbard_square_lattice_torch(Lx, Ly, t, U, N_f, pbc=False, n_fermions_per_spin=n_fermions_per_spin)
graph = H.graph
# TN parameters
D = int(4)
chi = int(4*D)
dtype=torch.float64

# Load PEPS
appendix = ''
if symmetry == 'U1_Z2':
    symmetry = 'Z2'
    appendix = '_U1'
elif symmetry == 'U1SU':
    symmetry = 'Z2'
    appendix = '_U1SU'
skeleton = pickle.load(open(pwd+f"/{Lx}x{Ly}/t={t}_U={U}/N={N_f}/{symmetry}/D={D}/peps_skeleton{appendix}.pkl", "rb"))
peps_params = pickle.load(open(pwd+f"/{Lx}x{Ly}/t={t}_U={U}/N={N_f}/{symmetry}/D={D}/peps_su_params{appendix}.pkl", "rb"))
peps = qtn.unpack(peps_params, skeleton)
# Precondition the fPEPS
## 1. Sync the stored fermionic phases
for ts in peps.tensors:
    ts.data.phase_sync(inplace=True)
## 2. Scale the tensor elements
scale = 4.0
peps.apply_to_arrays(lambda x: torch.tensor(scale*x, dtype=dtype))
## 3. Set the exponent to 0.0
peps.exponent = 0.0
# correct the format of oddpos
for ts in peps.tensors:
    ts.data.phase_sync(inplace=True)
    # for Z2 fPEPS converted from U1 fPEPS, need to correct the format of oddpos
    if ts.data.oddpos:
        oddpos = ts.data.oddpos
        nested_oddpos = oddpos[0].label[0]
        if isinstance(nested_oddpos, FermionicOperator):
            ts.data._oddpos = (nested_oddpos,)

# VMC sample size
N_samples = int(1)
N_samples = closest_divisible(N_samples, SIZE)
if (N_samples/SIZE)%2 != 0:
    N_samples += SIZE

radius = 1
model = fTN_BFA_cluster_Model_reuse(
        peps,
        max_bond=chi,
        embedding_dim=16,
        attention_heads=4,
        nn_final_dim=D,
        nn_eta=1.0,
        radius=radius,
        jastrow=False,
        dtype=dtype,
)

# model1 = fTNModel_reuse(
#     peps
# )

init_std = float(model.get_tn_params_vec().abs().std().item())
model.apply(lambda x: init_weights_to_zero(x, std=init_std))

model_names = {
    fTNModel_reuse: f'fTN_reuse{appendix}',
    fTN_BFA_cluster_Model_reuse: f'fTNNN_r={radius}{appendix}',
}
model_name = model_names.get(type(model), 'UnknownModel')

In [59]:
init_std

0.09643970406198367

In [52]:
random_x = torch.tensor(H.hilbert.random_state())
tn_params_vec = model.get_tn_params_vec()

In [56]:
with torch.no_grad():
    tn_params_std = tn_params_vec .std()
    params_std = model.from_params_to_vec().std()

    print(tn_params_std, params_std)
    print(model.get_tn_params_vec().abs().min(), model.get_tn_params_vec().abs().max())

tensor(0.0992, dtype=torch.float64) tensor(0.0520, dtype=torch.float64)
tensor(4.4639e-32, dtype=torch.float64) tensor(2.1899, dtype=torch.float64)


In [57]:
tn_params_vec.abs().std(), tn_params_vec.abs().min()

(tensor(0.0964, dtype=torch.float64, grad_fn=<StdBackward0>),
 tensor(4.4639e-32, dtype=torch.float64, grad_fn=<MinBackward1>))

In [47]:
nn_params_vec = []
for x, y in model.named_parameters():
    if x[:5] == 'torch':
        continue
    nn_params_vec.append(y.reshape(-1))

nn_params_vec = torch.cat(nn_params_vec, dim=0)
nn_params_vec.std()

tensor(0.0427, dtype=torch.float64, grad_fn=<StdBackward0>)